# EEG-Flow — 8. Epochs and evoked responses

Creates stimulus-locked and response-locked epochs and evoked responses
from the step7 preprocessed raw. Runs in parallel across participants.

**Stim-locked** (all cohorts, UT and oddball):
- Window: −200 to 800ms, baseline: −200 to 0ms
- All three event types (standard, target, novel) in a single file
- Epochs immediately following a response are dropped; incorrect trials
  are dropped. Metadata computed with `make_metadata_mne` (0–0.999s window)
- `_step8_stimlocked-epo.fif`, `_step8_stimlocked-ave.fif`

**Response-locked** (active cohort only, UT and oddball):
- Hits only (correct target responses with RT ≥ 200ms)
- Window: −200 to 800ms; two baseline corrections saved separately:
  - bc −200 to −100ms, all channels (load paper):
    `_step8_responselocked_bc-200-100-epo.fif / -ave.fif`
  - bc −200 to 0ms + CSD, EEG only (flow paper / Castellar replication):
    `_step8_responselocked_CSD_bc-200-0-epo.fif / -ave.fif`

**v2 changes vs v1 step70dd:**
- All event types are in a single stim-locked file (v1 had one file per type)
- EOG channels are included in stim-locked and non-CSD response-locked outputs
- "Epoch after response" drop logic is corrected: v1 computed drop indices
  from the post-bad-segment epoch sequence causing index-shift artifacts;
  v2 uses original metadata indices. Epoch count differences are small
  (< 2%) and v2 is more accurate
- No PTP autoreject — bad segment annotations from step3 handle rejection

Last edit: 14.10.2026 @anguyen

In [ ]:
import time
from itertools import product
import multiprocessing as mp

from mne import set_log_level as set_log_level_mne

from eeg_flow import set_log_level
from eeg_flow.utils import parallel
from eeg_flow.tasks import create_epochs_evoked

The parameters of the file to process are defined below. Locks are created to prevent someone else from running the same task and from writing the same derivatives.

In [ ]:
set_log_level_mne("WARNING")
set_log_level("INFO")

In [ ]:
PARTICIPANTS_WITH_GROUPS: list[str] = []  # list of str "Pxx-Gy", e.g. ["P02-G2"]
TASKS: list[str]                    = []  # ["oddball"], ["UT"] or ["oddball", "UT"]
RUNS: list[int]                     = []  # [1], [2] or [1, 2]

inputs = [
    (t[0].split("-") + list(t[1:]))
    for t in product(PARTICIPANTS_WITH_GROUPS, TASKS, RUNS)
]
print(inputs)

In [ ]:
%%time
current_time = time.strftime("%H:%M:%S")
print("Start time", current_time)

assert len(inputs) != 0  # sanity-check
# each job will need 1 threads and mp.cpu_count() usually returns
# the number of threads (usually 2 per core), thus let's spawn 2
# process per core maximum.
n_jobs = min(len(inputs), (mp.cpu_count() - 2))
parallel(create_epochs_evoked, n_jobs, inputs)